# 01 - Self-Attention: Intuition and Math

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain the **attention** concept: "which parts of the input should I focus on?"
- Derive and implement **Query, Key, Value** matrices from an input embedding
- Implement **scaled dot-product attention** from scratch in PyTorch
- Understand why we scale by $\sqrt{d_k}$ (preventing softmax saturation)
- Apply **causal masking** (decoder) and **padding masking**
- Implement **multi-head attention** from scratch
- Visualize attention weights as heatmaps

## Prerequisites

- Comfortable with PyTorch tensors, `nn.Module`, and matrix multiplication
- Basic understanding of softmax and neural network layers
- Familiarity with sequence-to-sequence concepts (helpful but not required)

## Table of Contents

1. [The Attention Concept](#1-the-attention-concept)
2. [Query, Key, Value Matrices](#2-query-key-value-matrices)
3. [Scaled Dot-Product Attention](#3-scaled-dot-product-attention)
4. [Why Scale by sqrt(d_k)?](#4-why-scale-by-sqrtd_k)
5. [Masking: Causal and Padding](#5-masking-causal-and-padding)
6. [Multi-Head Attention](#6-multi-head-attention)
7. [Code: Implement Scaled Dot-Product Attention](#7-code-implement-scaled-dot-product-attention)
8. [Code: Implement Multi-Head Attention](#8-code-implement-multi-head-attention)
9. [Code: Visualize Attention Weights](#9-code-visualize-attention-weights)
10. [Exercise: Implement Masked Attention](#10-exercise-implement-masked-attention)
11. [Common Mistakes & Debugging Tips](#11-common-mistakes--debugging-tips)

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

set_global_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

print(f"PyTorch version: {torch.__version__}")

---
## 1. The Attention Concept

**Core idea:** When processing a sequence (e.g., a sentence), not all parts are equally relevant to each other. Attention lets the model **dynamically decide** which parts of the input to focus on.

**Analogy:** Reading a sentence like *"The cat sat on the mat because it was tired."*
- To understand what *"it"* refers to, you need to **attend** to *"cat"*, not *"mat"*.
- A fixed-size hidden state (like in vanilla RNNs) would have to cram all context into one vector.
- Attention creates **direct connections** between every pair of positions.

**Why attention matters:**
- RNNs process tokens sequentially -- information about early tokens degrades over long sequences
- Attention allows **parallel processing** and **direct access** to any position
- Self-attention: each token attends to **all tokens in the same sequence** (including itself)

**Self-attention in one sentence:** For each token, compute a weighted average of all tokens' representations, where the weights reflect relevance.

---
## 2. Query, Key, Value Matrices

Self-attention uses three learned linear projections to transform the input:

Given input $X \in \mathbb{R}^{n \times d}$ (sequence of $n$ tokens, each with $d$-dimensional embedding):

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

where:
- $W_Q \in \mathbb{R}^{d \times d_k}$ -- **Query** projection
- $W_K \in \mathbb{R}^{d \times d_k}$ -- **Key** projection  
- $W_V \in \mathbb{R}^{d \times d_v}$ -- **Value** projection

**Intuition (library analogy):**
- **Query (Q):** "What am I looking for?" -- each token's question
- **Key (K):** "What do I contain?" -- each token's label/description
- **Value (V):** "What information do I carry?" -- the actual content to retrieve

The attention mechanism computes **similarity between queries and keys** to decide how much of each value to include.

In [ ]:
# Demonstrate Q, K, V computation
set_global_seed(42)

# Simulate a sequence of 4 tokens, each with embedding dim 8
seq_len, d_model = 4, 8
d_k = d_v = 8  # often d_k = d_v = d_model

X = torch.randn(seq_len, d_model)  # (4, 8)
print(f"Input X shape: {X.shape}")

# Learned projection matrices
W_Q = nn.Linear(d_model, d_k, bias=False)
W_K = nn.Linear(d_model, d_k, bias=False)
W_V = nn.Linear(d_model, d_v, bias=False)

Q = W_Q(X)  # (4, 8)
K = W_K(X)  # (4, 8)
V = W_V(X)  # (4, 8)

print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")
print()
print("Each row is one token's query/key/value vector.")

---
## 3. Scaled Dot-Product Attention

The core attention formula:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**Step-by-step breakdown:**

1. **Compute attention scores:** $S = QK^T \in \mathbb{R}^{n \times n}$
   - $S_{ij}$ = dot product between query $i$ and key $j$ = how much token $i$ should attend to token $j$

2. **Scale:** Divide by $\sqrt{d_k}$ to prevent scores from becoming too large

3. **Softmax:** Convert scores to probabilities (each row sums to 1)
   - $\alpha_{ij} = \text{softmax}(S_i / \sqrt{d_k})_j$

4. **Weighted sum of values:** Output$_i = \sum_j \alpha_{ij} V_j$

In [ ]:
# Walk through each step manually
set_global_seed(42)

seq_len, d_k = 4, 8
Q = torch.randn(seq_len, d_k)
K = torch.randn(seq_len, d_k)
V = torch.randn(seq_len, d_k)

# Step 1: Compute raw attention scores
scores = Q @ K.T  # (4, 4)
print("Step 1 - Raw attention scores (Q @ K^T):")
print(scores)
print(f"Shape: {scores.shape}")
print()

# Step 2: Scale
scale = math.sqrt(d_k)
scaled_scores = scores / scale
print(f"Step 2 - Scaled by sqrt({d_k}) = {scale:.2f}:")
print(scaled_scores)
print()

# Step 3: Softmax (row-wise)
attn_weights = F.softmax(scaled_scores, dim=-1)  # (4, 4)
print("Step 3 - Attention weights (softmax):")
print(attn_weights)
print(f"Row sums: {attn_weights.sum(dim=-1)}")
print()

# Step 4: Weighted sum of values
output = attn_weights @ V  # (4, 4) @ (4, 8) = (4, 8)
print(f"Step 4 - Output shape: {output.shape}")
print("Each row is a weighted combination of all value vectors.")

---
## 4. Why Scale by $\sqrt{d_k}$?

**Problem:** When $d_k$ is large, dot products $q \cdot k$ tend to have large magnitude. This pushes softmax into regions where gradients are extremely small (saturation).

**Math:** If $q_i$ and $k_i$ are independent random variables with mean 0 and variance 1:
$$\text{Var}(q \cdot k) = \text{Var}\left(\sum_{i=1}^{d_k} q_i k_i\right) = d_k$$

So the standard deviation of the dot product grows as $\sqrt{d_k}$. Dividing by $\sqrt{d_k}$ restores unit variance.

**Experiment:** Let us verify this empirically.

In [ ]:
set_global_seed(42)

dims = [8, 64, 128, 512, 1024]
n_samples = 10000

print(f"{'d_k':>6} | {'Mean dot':>10} | {'Std dot':>10} | {'Std/sqrt(d_k)':>14} | {'Softmax max (unscaled)':>22} | {'Softmax max (scaled)':>22}")
print("-" * 100)

for d in dims:
    q = torch.randn(n_samples, d)
    k = torch.randn(n_samples, d)
    
    dots = (q * k).sum(dim=-1)  # dot products
    
    # Softmax over a few random keys for one query
    q_single = torch.randn(1, d)
    k_batch = torch.randn(10, d)
    scores_unscaled = q_single @ k_batch.T
    scores_scaled = scores_unscaled / math.sqrt(d)
    
    sm_unscaled = F.softmax(scores_unscaled, dim=-1)
    sm_scaled = F.softmax(scores_scaled, dim=-1)
    
    print(f"{d:>6} | {dots.mean():>10.3f} | {dots.std():>10.3f} | {dots.std()/math.sqrt(d):>14.3f} | {sm_unscaled.max():>22.4f} | {sm_scaled.max():>22.4f}")

print()
print("Observation: Without scaling, softmax becomes peaky (max near 1.0) for large d_k.")
print("Scaling restores more uniform distributions, enabling meaningful gradients.")

---
## 5. Masking: Causal and Padding

### Causal Mask (Decoder)

In autoregressive models (like GPT), token $i$ should only attend to tokens $\leq i$ (no peeking at future tokens).

We achieve this by setting future positions to $-\infty$ **before** softmax:

$$\text{mask}_{ij} = \begin{cases} 0 & \text{if } j \leq i \\ -\infty & \text{if } j > i \end{cases}$$

After softmax, $e^{-\infty} = 0$, so future tokens get zero attention weight.

### Padding Mask

When batching sequences of different lengths, shorter sequences are padded (e.g., with 0). We mask out padding tokens so they do not receive attention.

In [ ]:
# Causal mask
seq_len = 5

# Create lower-triangular mask (True = attend, False = mask out)
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
print("Causal mask (lower triangular):")
print(causal_mask.int())
print()

# Apply to attention scores
set_global_seed(42)
scores = torch.randn(seq_len, seq_len)
print("Raw scores:")
print(scores.round(decimals=2))
print()

# Set masked positions to -inf
masked_scores = scores.masked_fill(~causal_mask, float('-inf'))
print("After causal masking:")
print(masked_scores.round(decimals=2))
print()

# Softmax: -inf positions become 0
attn_weights = F.softmax(masked_scores, dim=-1)
print("Attention weights after softmax:")
print(attn_weights.round(decimals=3))
print()
print("Notice: Each row only has non-zero weights for positions <= its index.")

In [ ]:
# Padding mask example
# Suppose we have a batch of 2 sequences with lengths [4, 2], padded to length 4
# Token IDs (0 = padding)
tokens = torch.tensor([
    [5, 12, 8, 3],  # full sequence, length 4
    [7, 2, 0, 0],   # actual length 2, padded with 0
])

# Padding mask: True where tokens are real, False where padding
padding_mask = (tokens != 0)  # (batch, seq_len)
print("Padding mask:")
print(padding_mask.int())
print()

# For attention: expand to (batch, 1, 1, seq_len) so it broadcasts with scores (batch, heads, seq, seq)
# Mask out columns corresponding to padding keys
attn_mask = padding_mask.unsqueeze(1).unsqueeze(2)  # (2, 1, 1, 4)
print(f"Attention mask shape: {attn_mask.shape}")
print("This broadcasts to mask out padding keys for all query positions and heads.")

---
## 6. Multi-Head Attention

Instead of performing a single attention function, we split the queries, keys, and values into $h$ **heads**, perform attention independently on each, then concatenate:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W_O$$

where each head is:

$$\text{head}_i = \text{Attention}(QW_Q^i, KW_K^i, VW_V^i)$$

**Why multiple heads?**
- Different heads can learn **different types of relationships** (e.g., syntactic, semantic, positional)
- Head 1 might learn "attend to the previous word"
- Head 2 might learn "attend to the verb of the sentence"
- Each head operates on a **lower-dimensional** subspace: $d_k = d_{\text{model}} / h$

**Dimensions:**
- Input: $(\text{batch}, \text{seq\_len}, d_{\text{model}})$
- Per-head: $(\text{batch}, \text{seq\_len}, d_k)$ where $d_k = d_{\text{model}} / h$
- After concatenation: $(\text{batch}, \text{seq\_len}, d_{\text{model}})$
- Output projection $W_O$: $(d_{\text{model}}, d_{\text{model}})$

---
## 7. Code: Implement Scaled Dot-Product Attention

Let us implement the full scaled dot-product attention function from scratch.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute scaled dot-product attention.
    
    Args:
        Q: Queries, shape (..., seq_len_q, d_k)
        K: Keys, shape (..., seq_len_k, d_k)
        V: Values, shape (..., seq_len_k, d_v)
        mask: Optional boolean mask, shape broadcastable to (..., seq_len_q, seq_len_k)
              True = attend, False = mask out
    
    Returns:
        output: Weighted sum of values, shape (..., seq_len_q, d_v)
        attn_weights: Attention weights, shape (..., seq_len_q, seq_len_k)
    """
    d_k = Q.size(-1)
    
    # Step 1: Compute attention scores
    scores = torch.matmul(Q, K.transpose(-2, -1))  # (..., seq_q, seq_k)
    
    # Step 2: Scale
    scores = scores / math.sqrt(d_k)
    
    # Step 3: Apply mask (if provided)
    if mask is not None:
        scores = scores.masked_fill(~mask, float('-inf'))
    
    # Step 4: Softmax
    attn_weights = F.softmax(scores, dim=-1)
    
    # Step 5: Weighted sum of values
    output = torch.matmul(attn_weights, V)  # (..., seq_q, d_v)
    
    return output, attn_weights


# Test it
set_global_seed(42)
seq_len, d_k = 4, 8

Q = torch.randn(seq_len, d_k)
K = torch.randn(seq_len, d_k)
V = torch.randn(seq_len, d_k)

output, attn_weights = scaled_dot_product_attention(Q, K, V)

print(f"Q shape: {Q.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"\nAttention weights (rows sum to 1):")
print(attn_weights.round(decimals=3))
print(f"Row sums: {attn_weights.sum(dim=-1).round(decimals=3)}")

In [ ]:
# Test with causal mask
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
output_masked, attn_masked = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

print("Causal-masked attention weights:")
print(attn_masked.round(decimals=3))
print("\nFuture positions have zero weight (lower triangular pattern).")

---
## 8. Code: Implement Multi-Head Attention

Now let us build a full multi-head attention module as a PyTorch `nn.Module`.

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention from scratch.
    
    Args:
        d_model: Embedding dimension
        n_heads: Number of attention heads
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # dimension per head
        
        # Projection layers (combined for all heads)
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)  # output projection
    
    def forward(self, Q, K, V, mask=None):
        """
        Args:
            Q: (batch, seq_len_q, d_model)
            K: (batch, seq_len_k, d_model)
            V: (batch, seq_len_k, d_model)
            mask: Optional, broadcastable to (batch, 1, seq_len_q, seq_len_k)
        
        Returns:
            output: (batch, seq_len_q, d_model)
            attn_weights: (batch, n_heads, seq_len_q, seq_len_k)
        """
        batch_size = Q.size(0)
        
        # 1. Linear projections
        Q = self.W_Q(Q)  # (batch, seq_q, d_model)
        K = self.W_K(K)  # (batch, seq_k, d_model)
        V = self.W_V(V)  # (batch, seq_k, d_model)
        
        # 2. Split into multiple heads
        # Reshape: (batch, seq, d_model) -> (batch, seq, n_heads, d_k) -> (batch, n_heads, seq, d_k)
        Q = Q.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        # 3. Scaled dot-product attention for each head
        # Q, K, V are now (batch, n_heads, seq, d_k)
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask=mask)
        
        # 4. Concatenate heads
        # (batch, n_heads, seq, d_k) -> (batch, seq, n_heads, d_k) -> (batch, seq, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        # 5. Output projection
        output = self.W_O(attn_output)  # (batch, seq, d_model)
        
        return output, attn_weights


# Test multi-head attention
set_global_seed(42)

batch_size, seq_len, d_model, n_heads = 2, 6, 16, 4

mha = MultiHeadAttention(d_model=d_model, n_heads=n_heads)
X = torch.randn(batch_size, seq_len, d_model)

# Self-attention: Q = K = V = X
output, attn_weights = mha(X, X, X)

print(f"Input shape:            {X.shape}")
print(f"Output shape:           {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"  - batch_size: {batch_size}")
print(f"  - n_heads: {n_heads}")
print(f"  - seq_len x seq_len: {seq_len} x {seq_len}")
print(f"  - d_k per head: {d_model // n_heads}")

In [ ]:
# Verify parameter count
total_params = sum(p.numel() for p in mha.parameters())
print(f"Total parameters: {total_params}")
print(f"  W_Q: {d_model} x {d_model} = {d_model * d_model}")
print(f"  W_K: {d_model} x {d_model} = {d_model * d_model}")
print(f"  W_V: {d_model} x {d_model} = {d_model * d_model}")
print(f"  W_O: {d_model} x {d_model} = {d_model * d_model}")
print(f"  Total: 4 x {d_model}^2 = {4 * d_model * d_model}")
assert total_params == 4 * d_model * d_model

---
## 9. Code: Visualize Attention Weights

Let us visualize what the attention weights look like for a sample "sentence" (using token names for clarity).

In [ ]:
def plot_attention_heatmap(attn_weights, tokens_q, tokens_k, title="Attention Weights", ax=None):
    """
    Plot attention weights as a heatmap.
    
    Args:
        attn_weights: (seq_len_q, seq_len_k) tensor
        tokens_q: list of query token labels
        tokens_k: list of key token labels
        title: plot title
        ax: optional matplotlib axis
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    
    weights = attn_weights.detach().cpu().numpy()
    im = ax.imshow(weights, cmap='Blues', vmin=0, vmax=1)
    
    ax.set_xticks(range(len(tokens_k)))
    ax.set_xticklabels(tokens_k, rotation=45, ha='right')
    ax.set_yticks(range(len(tokens_q)))
    ax.set_yticklabels(tokens_q)
    
    # Annotate cells
    for i in range(len(tokens_q)):
        for j in range(len(tokens_k)):
            color = 'white' if weights[i, j] > 0.5 else 'black'
            ax.text(j, i, f'{weights[i,j]:.2f}', ha='center', va='center',
                   color=color, fontsize=9)
    
    ax.set_xlabel('Keys')
    ax.set_ylabel('Queries')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return ax


# Create a sample sentence and compute attention
set_global_seed(42)

tokens = ["The", "cat", "sat", "on", "the", "mat"]
seq_len = len(tokens)
d_model = 16
n_heads = 4

# Simulate embeddings (in practice these come from an embedding layer)
X = torch.randn(1, seq_len, d_model)

mha = MultiHeadAttention(d_model=d_model, n_heads=n_heads)
output, attn_weights = mha(X, X, X)

# Plot attention for each head
fig, axes = plt.subplots(1, n_heads, figsize=(5 * n_heads, 5))
for h in range(n_heads):
    # attn_weights: (batch=1, heads, seq_q, seq_k)
    plot_attention_heatmap(
        attn_weights[0, h],  # first batch, head h
        tokens, tokens,
        title=f"Head {h+1}",
        ax=axes[h]
    )

plt.suptitle("Multi-Head Attention Weights (randomly initialized)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Note: These are random (untrained) weights. After training, each head")
print("would learn different attention patterns (syntactic, semantic, etc.).")

In [ ]:
# Average attention across heads
avg_attn = attn_weights[0].mean(dim=0)  # (seq, seq)

fig, ax = plt.subplots(figsize=(6, 5))
plot_attention_heatmap(avg_attn, tokens, tokens, title="Average Attention Across All Heads", ax=ax)
plt.tight_layout()
plt.show()

---
## 10. Exercise: Implement Masked Attention

**Task:** Modify the multi-head attention to support causal masking (autoregressive), then visualize the masked attention pattern.

1. Create a causal mask for a sequence of length 6
2. Pass it through `MultiHeadAttention` and visualize the attention heatmap
3. Verify that no token attends to future positions

```python
# Your code here

# Step 1: Create causal mask
# Hint: torch.tril + unsqueeze for broadcasting to (batch, 1, seq, seq)
# causal_mask = ...

# Step 2: Run MHA with mask
# output_masked, attn_masked = mha(X, X, X, mask=causal_mask)

# Step 3: Visualize and verify
# ...
```

In [ ]:
# Try the exercise yourself before looking at the solution!








### Solution

In [ ]:
# ----- Solution -----
set_global_seed(42)

tokens = ["The", "cat", "sat", "on", "the", "mat"]
seq_len = len(tokens)
d_model = 16
n_heads = 4

X = torch.randn(1, seq_len, d_model)
mha = MultiHeadAttention(d_model=d_model, n_heads=n_heads)

# Step 1: Create causal mask
# Shape: (1, 1, seq_len, seq_len) to broadcast with (batch, n_heads, seq, seq)
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq, seq)
print(f"Causal mask shape: {causal_mask.shape}")

# Step 2: Run MHA with mask
output_masked, attn_masked = mha(X, X, X, mask=causal_mask)

# Step 3: Visualize
fig, axes = plt.subplots(1, n_heads, figsize=(5 * n_heads, 5))
for h in range(n_heads):
    plot_attention_heatmap(
        attn_masked[0, h], tokens, tokens,
        title=f"Head {h+1} (Causal Masked)",
        ax=axes[h]
    )
plt.suptitle("Causal-Masked Multi-Head Attention", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Step 4: Verify no future attention
for h in range(n_heads):
    upper_tri = attn_masked[0, h].triu(diagonal=1)
    assert upper_tri.sum().item() == 0, f"Head {h} attends to future!"
print("Verified: No head attends to future positions.")

---
## 11. Common Mistakes & Debugging Tips

**1. Forgetting to scale by $\sqrt{d_k}$**
- Symptom: Softmax outputs are nearly one-hot (one value near 1.0, rest near 0.0)
- Fix: Always divide scores by `math.sqrt(d_k)` before softmax

**2. Wrong mask shape / broadcasting**
- Attention scores have shape `(batch, n_heads, seq_q, seq_k)`
- Mask must be broadcastable to this shape
- Common shapes: `(1, 1, seq, seq)` for causal, `(batch, 1, 1, seq)` for padding
- Tip: Use `mask.unsqueeze()` to add dimensions for broadcasting

**3. Masking with 0 instead of $-\infty$**
- Setting masked scores to 0 does NOT zero out their softmax probability
- Must use `float('-inf')` so that `softmax(-inf) = 0`

**4. Transpose confusion in multi-head split**
- After `view(batch, seq, n_heads, d_k)`, you need `.transpose(1, 2)` to get `(batch, n_heads, seq, d_k)`
- Forgetting this transposes heads and sequence dimensions, producing wrong results

**5. Not using `.contiguous()` before `.view()`**
- After transpose, the tensor memory layout may not be contiguous
- `.view()` requires contiguous memory; use `.contiguous().view()` or `.reshape()`

**6. Q, K, V dimension mismatch**
- Q and K must have the same last dimension ($d_k$) for the dot product
- V can have a different last dimension ($d_v$), but typically $d_v = d_k$

**7. NaN in attention weights**
- If ALL positions in a row are masked, softmax gets `softmax([-inf, -inf, ...])` = NaN
- Fix: Ensure at least one position is unmasked, or handle this edge case

---

**Next notebook:** [02 - Transformer Architecture: Encoder-Decoder](02_Transformer_Architecture_Encoder_Decoder.ipynb) -- we assemble attention into full Transformer encoder and decoder blocks with positional encoding, layer normalization, and residual connections.